# Gnasher Group | V2 inference only

Load the original v2_run04 checkpoint and locked validation threshold. Colab CPU or GPU works with compatible Torch/Torchvision installed. This notebook requires no training or dataset download. Keep the checkpoint in your Drive or download it from its owner and upload it when prompted. Only load course checkpoint files from a trusted source.

In [ ]:
import hashlib
import json
from pathlib import Path
from urllib.request import urlretrieve

import torch
from google.colab import files
from IPython.display import display
from PIL import Image, UnidentifiedImageError
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

EXPECTED_SHA = "946a3fddca5e3955e58d0395148b7e099c65cb0ff91ebdda15b5f6159b88041c"
GROUPS = ["Sporting", "Hound", "Working", "Terrier", "Toy", "Non-Sporting", "Herding"]
CHECKPOINT = Path("/content/best.pt")  # Or mount Drive and change this path.
THRESHOLD_URL = "https://raw.githubusercontent.com/IreneAlexRamos/itai1378-project/main/results/v2_run04/threshold.json"
threshold_path = Path("/content/gnasher_threshold.json")
urlretrieve(THRESHOLD_URL, threshold_path)
selection = json.loads(threshold_path.read_text())
if not CHECKPOINT.is_file():
    uploaded = files.upload()
    if set(uploaded) != {"best.pt"}:
        raise ValueError("Upload only best.pt from v2_run04.")
    CHECKPOINT.write_bytes(uploaded["best.pt"])

if selection["checkpoint_sha256"] != EXPECTED_SHA or hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() != EXPECTED_SHA:
    raise ValueError("This checkpoint and threshold do not match the recorded v2_run04 run.")
threshold = float(selection["threshold"])
if not 0 <= threshold <= 1:
    raise ValueError("Invalid threshold.")
checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
if checkpoint["class_names"] != GROUPS:
    raise ValueError("Checkpoint class order does not match the seven AKC groups.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = efficientnet_b0(weights=None)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, len(GROUPS))
model.load_state_dict(checkpoint["state_dict"])
model = model.to(device).eval()
preprocess = EfficientNet_B0_Weights.DEFAULT.transforms()
print(f"v2_run04 ready on {device}. Locked threshold: {threshold:.0%}.")

Run the next cell for each new dog photo. The score is a model output, and the system has no non-dog detector. Below the locked threshold, it displays both candidates and abstains.

In [ ]:
from io import BytesIO

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one dog photo.")
name, image_bytes = next(iter(uploaded.items()))
try:
    with Image.open(BytesIO(image_bytes)) as source:
        photo = source.convert("RGB")
except (UnidentifiedImageError, OSError) as error:
    raise ValueError("Choose a readable image file.") from error

with torch.inference_mode():
    batch = preprocess(photo).unsqueeze(0).to(device)
    scores, indices = model(batch).softmax(dim=1)[0].topk(2)
    candidates = [(GROUPS[i], score) for i, score in zip(indices.tolist(), scores.tolist())]

preview = photo.copy()
preview.thumbnail((640, 480))
display(preview)
print(name)
leading_group, leading_score = candidates[0]
print(f"AKC group: {leading_group}" if leading_score >= threshold else "AKC group: Uncertain")
print(f"Leading model score: {leading_score:.1%}")
for rank, (group, score) in enumerate(candidates, 1):
    print(f"{rank}. {group}: {score:.1%}")
if leading_score < threshold:
    print("Uncertainty warning: the leading score is below the locked threshold.")
print("Use dog photos only. These scores do not verify breed or detect non-dogs.")